# When Models Lie to Please: Google Colab Runner

This notebook runs the actual, GPU-heavy 4-phase sparse interpretability research pipeline on a high-VRAM Google Colab runtime (T4, L4, or A100 GPU).

### Pipeline Steps:
1. **Git Clone/Pull**: Sets up the codebase and installs all dependencies.
2. **Build Datasets**: Generates the paired logic/opinion/sycophancy datasets.
3. **Phase 1 (Feature Discovery)**: Extracts activations and finds differential SAE features.
4. **Phase 2 (Circuit Tracing)**: Builds transcoder attribution graphs and analyzes divergence.
5. **Phase 3 (Transfer Testing)**: Runs cross-condition transfer and measures representation geometry.
6. **Phase 4 (Interventions & CAST)**: Trains classifiers and tests steering/CAST mitigations.
7. **Zip & Download**: Compresses results and datasets into a ZIP file for local download.

## Step 1: Install Dependencies and Setup

In [ ]:
# Clone repo or pull latest updates
!git clone https://github.com/ashioyajotham/when_models_lie_to_please.git || (cd when_models_lie_to_please && git pull)
%cd when_models_lie_to_please

# Install dependencies
!pip install -e ".[dev]"
!pip install git+https://github.com/google-deepmind/mishax.git

print("Setup complete!")

## Step 2: Configure Token and Run Pipeline
Make sure to add your HuggingFace token as a secret in Colab (named `HF_TOKEN`) or enter it when prompted.

In [ ]:
import os
import glob
import json
import time

# Authenticate with HuggingFace (required for gated Gemma 3 models)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = input("Enter your HuggingFace Token: ").strip()

pipeline_start = time.time()

# Step 0: Build paired prompt datasets
!python experiments/scripts/build_datasets.py --output-dir data/processed --min-pairs 500

# Clear any old/corrupted activation cache to ensure fresh extraction with loaded SAEs
!rm -rf data/activations/gemma3_4b

# Step 1: Feature discovery
!python experiments/scripts/run_phase1.py --config configs/experiment_configs/phase1_features.yaml --ignore-cache

# Find latest Phase 1 run_id to pass into Phase 2/3/4
phase1_runs = sorted(glob.glob("experiments/results/phase1/*"))
if not phase1_runs:
    raise RuntimeError("Phase 1 execution did not produce any output folders.")
run_id = os.path.basename(phase1_runs[-1])
print(f"\n>>> Detected Phase 1 Run ID: {run_id} <<<\n")

# Step 2: Circuit tracing
!python experiments/scripts/run_phase2.py --config configs/experiment_configs/phase2_circuits.yaml --phase1-run {run_id}

# Step 3: Shared mechanism testing
!python experiments/scripts/run_phase3.py --config configs/experiment_configs/phase3_transfer.yaml --phase1-run {run_id}

# Step 4: Detection and mitigation
!python experiments/scripts/run_phase4.py --config configs/experiment_configs/phase4_interventions.yaml --phase1-run {run_id} --phase3-run {run_id}

# ═══════════════════════════════════════════════════════════════════════
# PIPELINE SUMMARY — always printed at the very end for visibility
# ═══════════════════════════════════════════════════════════════════════
elapsed = time.time() - pipeline_start
print("\n")
print("=" * 72)
print("  PIPELINE EXECUTION SUMMARY")
print("=" * 72)
print(f"  Total elapsed time: {elapsed/60:.1f} minutes")
print("-" * 72)

for phase_name in ["phase1", "phase2", "phase3", "phase4"]:
    base = f"experiments/results/{phase_name}"
    if not os.path.exists(base):
        print(f"\n  [MISSING] {phase_name.upper()}: No results directory found")
        continue
    runs = sorted(os.listdir(base))
    if not runs:
        print(f"\n  [MISSING] {phase_name.upper()}: Empty results directory")
        continue
    latest = runs[-1]
    summary_path = os.path.join(base, latest, "run_summary.json")
    if not os.path.exists(summary_path):
        print(f"\n  [WARNING] {phase_name.upper()} (run {latest}): No run_summary.json (may have crashed)")
        partial = os.listdir(os.path.join(base, latest))
        if partial:
            print(f"      Partial files: {', '.join(partial[:10])}")
        continue
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"\n  [OK] {phase_name.upper()} (run {latest})")
    for k, v in summary.items():
        if isinstance(v, dict):
            print(f"      {k}:")
            for kk, vv in v.items():
                print(f"        {kk}: {vv}")
        elif isinstance(v, list):
            print(f"      {k}: {', '.join(str(x) for x in v)}")
        else:
            print(f"      {k}: {v}")

print("\n" + "=" * 72)
print("  Pipeline execution finished!")
print("=" * 72)
print("\nRun Step 3 below to zip and download results.")

## Step 3: Zip and Download Results

In [ ]:
import zipfile
import datetime
from google.colab import files

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"colab_results_{timestamp}.zip"

print(f"Packaging results into {zip_filename}...")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Compress experiments results
    for root, dirs, files_in_dir in os.walk("experiments/results"):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file))
    
    # Compress processed datasets
    for root, dirs, files_in_dir in os.walk("data/processed"):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file))

print("Zipping finished. Triggering download in browser...")
files.download(zip_filename)